# test_prosumer.ipynb：逐行演算 `data/loaders/prosumer.py`

这个 notebook 用一份很小的 prosumer 示例数据，逐个测试并解释 `prosumer.py` 里的函数。

每个函数都会按同一种方式展开：

1. 给出一个具体的数学输入例子。
2. 按函数源码里的关键代码路径，一行一行打印中间计算结果。
3. 最后调用真实函数，展示最终输出。

带 `_` 的函数/方法是内部实现细节；这里直接调用它们只是为了学习和验证。

## 0. 准备环境和示例 CSV

`ProsumerDataset` 固定读取 `data_dir / processed/prosumer`。下面先造 4 个文件：

- `household.csv`：家庭基础负荷。
- `heatpump.csv`：热泵负荷。
- `pv_reference.csv`：参考 PV 曲线。
- `price.csv`：批发电价。

示例数值故意设计得很容易心算：

- `profile_a` 的 household 从 1.00 开始，每步加 0.01。
- `profile_a` 的 heatpump 从 0.20 开始，每步加 0.002。
- 所以某一行如果 step=8，则原始总负荷是 `1.08 + 0.216 = 1.296`。
- 如果 `load_scale=10`，则缩放后是 `12.96`。

In [1]:
from datetime import date
from pathlib import Path
import shutil
import sys


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "data").is_dir() and (candidate / "tests").is_dir():
            return candidate
    raise RuntimeError(f"Cannot locate project root from {start}")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from IPython.display import display

from data.loaders.prosumer import (
    DEFAULT_PROCESSED_SUBDIR,
    TZ_LOCAL,
    ProsumerDataset,
    _coerce_scale_vector,
    _parse_optional_local_date,
)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)


def as_list(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, pd.Series):
        return value.tolist()
    return value


def show_steps(title, input_text, rows, final_output=None):
    print()
    print(title)
    print(f"输入：{input_text}")
    display(pd.DataFrame(rows, columns=["步骤", "对应源码/表达式", "怎么算", "结果"]))
    if final_output is not None:
        print("最终输出：")
        display(final_output if isinstance(final_output, (pd.DataFrame, pd.Series)) else final_output)


DEMO_ROOT = Path("tmp/test_prosumer_demo")
PROCESSED_DIR = DEMO_ROOT / DEFAULT_PROCESSED_SUBDIR

if DEMO_ROOT.exists():
    shutil.rmtree(DEMO_ROOT)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

local_ts = pd.date_range("2019-12-31 22:00", periods=112, freq="15min", tz=TZ_LOCAL)
utc_ts = local_ts.tz_convert("UTC").astype(str)
step = np.arange(len(local_ts), dtype=np.float32)
minute_of_day = np.asarray(local_ts.hour * 60 + local_ts.minute, dtype=np.float32)
day_fraction = minute_of_day / np.float32(24 * 60)
solar_shape = np.maximum(0.0, np.sin(np.pi * (day_fraction - 0.25) / 0.5)).astype(np.float32)
price_wave = np.sin(2 * np.pi * day_fraction).astype(np.float32)

household = pd.DataFrame(
    {
        "timestamp": utc_ts,
        "profile_a": 1.00 + 0.010 * step,
        "profile_b": 1.50 + 0.015 * step,
    }
)
heatpump = pd.DataFrame(
    {
        "timestamp": utc_ts,
        "profile_a": 0.20 + 0.002 * step,
        "profile_b": 0.30 + 0.003 * step,
    }
)
pv_reference = pd.DataFrame(
    {
        "timestamp": utc_ts,
        "ref_east": solar_shape * 2.0,
        "ref_south": solar_shape * 3.0,
        "ref_west": solar_shape * 1.6,
    }
)
price = pd.DataFrame(
    {
        "timestamp": utc_ts,
        "price": 0.22 + 0.04 * price_wave,
    }
)

for name, frame in {
    "household.csv": household,
    "heatpump.csv": heatpump,
    "pv_reference.csv": pv_reference,
    "price.csv": price,
}.items():
    frame.to_csv(PROCESSED_DIR / name, index=False)

print(f"示例数据目录：{PROCESSED_DIR.resolve()}")
print(f"CSV 行数：{len(household)}")
print(f"本地时间范围：{local_ts[0]} -> {local_ts[-1]}")
display(household.head(3))
display(heatpump.head(3))

示例数据目录：D:\GithubProject\MADRL_ESS\tmp\test_prosumer_demo\processed\prosumer
CSV 行数：112
本地时间范围：2019-12-31 22:00:00+01:00 -> 2020-01-02 01:45:00+01:00


,timestamp,profile_a,profile_b
0,2019-12-31 21:00:00+00:00,1.00,1.500
1,2019-12-31 21:15:00+00:00,1.01,1.515
2,2019-12-31 21:30:00+00:00,1.02,1.530


,timestamp,profile_a,profile_b
0,2019-12-31 21:00:00+00:00,0.200,0.300
1,2019-12-31 21:15:00+00:00,0.202,0.303
2,2019-12-31 21:30:00+00:00,0.204,0.306


## 1. `_parse_optional_local_date(value)`

作用：把可选日期配置统一成本地自然日，后面才能用日期做过滤。

In [2]:
value = "2020-01-01 15:30"
condition = value in (None, "")
timestamp_value = pd.Timestamp(value)
date_value = timestamp_value.date()
actual = _parse_optional_local_date(value)

rows = [
    (1, "value in (None, '')", "检查是不是空日期配置", condition),
    (2, "pd.Timestamp(value)", "字符串转成 pandas 时间戳", repr(timestamp_value)),
    (3, ".date()", "丢掉小时分钟，只保留自然日", repr(date_value)),
    (4, "return ...", "非空输入走 Timestamp(value).date() 分支", repr(actual)),
]
show_steps("_parse_optional_local_date 的逐步演算", 'value="2020-01-01 15:30"', rows, actual)

empty_value = ""
empty_actual = _parse_optional_local_date(empty_value)
print(f"补充空值分支：输入空字符串 -> {empty_actual}")


_parse_optional_local_date 的逐步演算
输入：value="2020-01-01 15:30"


,步骤,对应源码/表达式,怎么算,结果
0,1,"value in (None, '')",检查是不是空日期配置,False
1,2,pd.Timestamp(value),字符串转成 pandas 时间戳,Timestamp('2020-01-01 15:30:00')
2,3,.date(),丢掉小时分钟，只保留自然日,"datetime.date(2020, 1, 1)"
3,4,return ...,非空输入走 Timestamp(value).date() 分支,"datetime.date(2020, 1, 1)"


最终输出：


datetime.date(2020, 1, 1)

补充空值分支：输入空字符串 -> None


## 2. `_coerce_scale_vector(scale, name, n_agents)`

作用：把缩放配置统一成每个智能体一个倍率。下面用一个列表例子演算。

In [14]:
scale = [1.0,1.0,1]
name = "demo_scale"
n_agents = 3

default = np.ones(n_agents, dtype=np.float32)
is_none = scale is None
is_scalar = isinstance(scale, (int, float, np.floating))
values = np.asarray(list(scale), dtype=np.float32).reshape(-1)
size_is_zero = values.size == 0
size_matches = values.size == n_agents
has_negative = bool(np.any(values < 0.0))
actual = _coerce_scale_vector(scale, name=name, n_agents=n_agents)
example_load = np.array([10.0, 20.0,10], dtype=np.float32)
scaled_load = example_load * actual

rows = [
    (1, "default = np.ones(n_agents)", "n_agents=2，所以默认倍率是两个 1", default.tolist()),
    (2, "scale is None", "scale=[1.0,0.5]，不是 None", is_none),
    (3, "isinstance(scale, scalar)", "列表不是标量，所以进入列表分支", is_scalar),
    (4, "np.asarray(list(scale)).reshape(-1)", "把 [1.0,0.5] 转成 float32 向量", values.tolist()),
    (5, "values.size == 0", "长度是 2，不是空列表", size_is_zero),
    (6, "values.size != n_agents", "2 == 2，长度匹配", not size_matches),
    (7, "np.any(values < 0.0)", "[1.0,0.5] 没有负数", has_negative),
    (8, "return values.astype(float32)", "返回倍率向量", actual.tolist()),
    (9, "example_load * scale", "如果原负荷是 [10,20]，缩放后是 [10*1.0,20*0.5]", scaled_load.tolist()),
]
show_steps("_coerce_scale_vector 的逐步演算", "scale=[1.0,0.5], n_agents=2；额外验证负荷 [10,20] 如何被缩放", rows, actual.tolist())


_coerce_scale_vector 的逐步演算
输入：scale=[1.0,0.5], n_agents=2；额外验证负荷 [10,20] 如何被缩放


,步骤,对应源码/表达式,怎么算,结果
0,1,default = np.ones(n_agents),n_agents=2，所以默认倍率是两个 1,"[1.0, 1.0, 1.0]"
1,2,scale is None,"scale=[1.0,0.5]，不是 None",False
2,3,"isinstance(scale, scalar)",列表不是标量，所以进入列表分支,False
3,4,np.asarray(list(scale)).reshape(-1),"把 [1.0,0.5] 转成 float32 向量","[1.0, 1.0, 1.0]"
4,5,values.size == 0,长度是 2，不是空列表,False
5,6,values.size != n_agents,2 == 2，长度匹配,False
6,7,np.any(values < 0.0),"[1.0,0.5] 没有负数",False
7,8,return values.astype(float32),返回倍率向量,"[1.0, 1.0, 1.0]"
8,9,example_load * scale,"如果原负荷是 [10,20]，缩放后是 [10*1.0,20*0.5]","[10.0, 20.0, 10.0]"


最终输出：


[1.0, 1.0, 1.0]

## 3. `ProsumerDataset.__init__(...)`

作用：创建数据集对象，并自动完成校验、加载、对齐和 episode 切片。

这里用真实 CSV 构造一个 dataset。重点看初始化过程中几个关键赋值是怎么得到的。

In [10]:
init_kwargs = dict(
    data_dir=DEMO_ROOT,
    episode_length=8,
    n_agents=2,
    agent_profiles=["profile_a", "profile_b"],
    year=2020,
    start_date="2020-01-01",
    end_date="2020-01-01",
    load_components=("household", "heatpump"),
    pv_reference="south",
    pv_capacity_kw=[5.0, 3.0],
    load_scale=[1.0, 10.0],
    pv_scale=[1.0, 1.0],
    history_warmup_steps=2,
    window_stride_steps=8,
    split="train",
)

manual_data_dir = Path(init_kwargs["data_dir"]) / DEFAULT_PROCESSED_SUBDIR
manual_episode_length = int(init_kwargs["episode_length"])
manual_stride = int(init_kwargs["window_stride_steps"] or init_kwargs["episode_length"])
manual_start = _parse_optional_local_date(init_kwargs["start_date"])
manual_end = _parse_optional_local_date(init_kwargs["end_date"])
manual_load_scale = _coerce_scale_vector(init_kwargs["load_scale"], name="load_scale", n_agents=2)
manual_pv_scale = _coerce_scale_vector(init_kwargs["pv_scale"], name="pv_scale", n_agents=2)

# 先用手算路径解释，再真实创建对象。
dataset = ProsumerDataset(**init_kwargs)

# 找到 2020-01-01 00:00 的那一行，验证一个具体负荷数学值。
first_active_ts = pd.Timestamp("2020-01-01 00:00", tz=TZ_LOCAL)
household_read_for_init = dataset._read_csv("household.csv")
heatpump_read_for_init = dataset._read_csv("heatpump.csv")
idx = int(household_read_for_init.index[household_read_for_init["timestamp"] == first_active_ts][0])
h_a = float(household_read_for_init.loc[idx, "profile_a"])
hp_a = float(heatpump_read_for_init.loc[idx, "profile_a"])
load_a = (h_a + hp_a) * manual_load_scale[0]
h_b = float(household_read_for_init.loc[idx, "profile_b"])
hp_b = float(heatpump_read_for_init.loc[idx, "profile_b"])
load_b = (h_b + hp_b) * manual_load_scale[1]

rows = [
    (1, "self.data_dir = Path(data_dir) / DEFAULT_PROCESSED_SUBDIR", f"{DEMO_ROOT} / {DEFAULT_PROCESSED_SUBDIR}", str(manual_data_dir)),
    (2, "self.episode_length = int(episode_length)", "int(8)", manual_episode_length),
    (3, "self.window_stride_steps = int(window_stride_steps or episode_length)", "window_stride_steps=8，所以 stride=8", manual_stride),
    (4, "self.start_date = _parse_optional_local_date(start_date)", '"2020-01-01" -> date(2020,1,1)', repr(manual_start)),
    (5, "self.end_date = _parse_optional_local_date(end_date)", '"2020-01-01" -> date(2020,1,1)', repr(manual_end)),
    (6, "self.load_scale = _coerce_scale_vector(...)", "[1.0,10.0] 已经是两个智能体的倍率", manual_load_scale.tolist()),
    (7, "self.pv_scale = _coerce_scale_vector(...)", "标量 1.0 复制给两个智能体", manual_pv_scale.tolist()),
    (8, "self._validate_init_args()", "episode_length=8>0，2 个画像列对应 2 个智能体", "通过"),
    (9, "self._load()", f"00:00 profile_a 总负荷=({h_a:.3f}+{hp_a:.3f})*1.0", round(load_a, 6)),
    (10, "self._load()", f"00:00 profile_b 总负荷=({h_b:.3f}+{hp_b:.3f})*10.0", round(load_b, 6)),
    (11, "self._num_episodes", "加载完成后 episode_slices 的数量", dataset.num_episodes()),
]
show_steps("ProsumerDataset.__init__ 的逐步演算", "见 init_kwargs；重点验证 2020-01-01 00:00 的负荷计算", rows)


ProsumerDataset.__init__ 的逐步演算
输入：见 init_kwargs；重点验证 2020-01-01 00:00 的负荷计算


,步骤,对应源码/表达式,怎么算,结果
0,1,self.data_dir = Path(data_dir) / DEFAULT_PROCESSED_SUBDIR,tmp\test_prosumer_demo / processed\prosumer,tmp\test_prosumer_demo\processed\prosumer
1,2,self.episode_length = int(episode_length),int(8),8
2,3,self.window_stride_steps = int(window_stride_steps or episode_length),window_stride_steps=8，所以 stride=8,8
3,4,self.start_date = _parse_optional_local_date(start_date),"""2020-01-01"" -> date(2020,1,1)","datetime.date(2020, 1, 1)"
4,5,self.end_date = _parse_optional_local_date(end_date),"""2020-01-01"" -> date(2020,1,1)","datetime.date(2020, 1, 1)"
5,6,self.load_scale = _coerce_scale_vector(...),"[1.0,10.0] 已经是两个智能体的倍率","[1.0, 10.0]"
6,7,self.pv_scale = _coerce_scale_vector(...),标量 1.0 复制给两个智能体,"[1.0, 1.0]"
7,8,self._validate_init_args(),episode_length=8>0，2 个画像列对应 2 个智能体,通过
8,9,self._load(),00:00 profile_a 总负荷=(1.080+0.216)*1.0,1.296
9,10,self._load(),00:00 profile_b 总负荷=(1.620+0.324)*10.0,19.440001


## 4. `ProsumerDataset._validate_init_args()`

作用：在入口处拒绝明显非法配置。这里先演算一个合法对象，再演示一个非法输入。

In [5]:
probe = object.__new__(ProsumerDataset)
probe.episode_length = 8
probe.window_stride_steps = 4
probe.history_warmup_steps = 2
probe.agent_profiles = ["profile_a", "profile_b"]
probe.n_agents = 2
probe.load_components = ("household", "heatpump")
probe.pv_capacity_kw = [5.0, 3.0]
probe.node_ids = [0, 1]

checks = []
for name in ("episode_length", "window_stride_steps"):
    value = getattr(probe, name)
    checks.append((name, value, value <= 0))
history_bad = probe.history_warmup_steps < 0
profiles_bad = len(probe.agent_profiles) != probe.n_agents
load_components_empty = not probe.load_components
pv_capacity_bad = probe.pv_capacity_kw is not None and len(probe.pv_capacity_kw) != probe.n_agents
node_ids_bad = len(probe.node_ids) != probe.n_agents
actual = probe._validate_init_args()

rows = [
    (1, "for name in ('episode_length','window_stride_steps')", "episode_length=8, stride=4，都大于 0", checks),
    (2, "history_warmup_steps < 0", "2 < 0 为 False", history_bad),
    (3, "len(agent_profiles) != n_agents", "len(['profile_a','profile_b'])=2，n_agents=2", profiles_bad),
    (4, "not self.load_components", "('household','heatpump') 非空", load_components_empty),
    (5, "len(pv_capacity_kw) != n_agents", "len([5.0,3.0])=2，n_agents=2", pv_capacity_bad),
    (6, "len(node_ids) != n_agents", "len([0,1])=2，n_agents=2", node_ids_bad),
    (7, "函数结束", "所有错误条件都是 False", actual),
]
show_steps("_validate_init_args 的合法配置演算", "episode_length=8, stride=4, n_agents=2, 两个画像列", rows, actual)

bad = object.__new__(ProsumerDataset)
bad.__dict__.update(probe.__dict__)
bad.episode_length = 0
try:
    bad._validate_init_args()
except Exception as exc:
    print(f"非法输入演示：episode_length=0 -> {type(exc).__name__}: {exc}")


_validate_init_args 的合法配置演算
输入：episode_length=8, stride=4, n_agents=2, 两个画像列


,步骤,对应源码/表达式,怎么算,结果
0,1,"for name in ('episode_length','window_stride_steps')","episode_length=8, stride=4，都大于 0","[(episode_length, 8, False), (window_stride_steps, 4, False)]"
1,2,history_warmup_steps < 0,2 < 0 为 False,False
2,3,len(agent_profiles) != n_agents,"len(['profile_a','profile_b'])=2，n_agents=2",False
3,4,not self.load_components,"('household','heatpump') 非空",False
4,5,len(pv_capacity_kw) != n_agents,"len([5.0,3.0])=2，n_agents=2",False
5,6,len(node_ids) != n_agents,"len([0,1])=2，n_agents=2",False
6,7,函数结束,所有错误条件都是 False,None


非法输入演示：episode_length=0 -> ValueError: episode_length must be positive, got 0


## 5. `ProsumerDataset._read_csv(name)`

作用：读取 CSV，并把 UTC 时间戳转为 `Europe/Berlin` 本地时间。

In [6]:
name = "household.csv"
path = dataset.data_dir / name
exists = path.exists()
raw_frame = pd.read_csv(path)
has_timestamp = "timestamp" in raw_frame.columns
raw_first = raw_frame.loc[0, "timestamp"]
converted_first = pd.to_datetime(raw_frame["timestamp"], utc=True).dt.tz_convert(TZ_LOCAL).iloc[0]
actual_frame = dataset._read_csv(name)

rows = [
    (1, "path = self.data_dir / name", f"{dataset.data_dir} / {name}", str(path)),
    (2, "if not path.exists()", "文件存在，所以不报 FileNotFoundError", exists),
    (3, "frame = pd.read_csv(path)", "读入前 1 行 timestamp", raw_first),
    (4, "if 'timestamp' not in frame.columns", "列名包含 timestamp", has_timestamp),
    (5, "pd.to_datetime(..., utc=True).dt.tz_convert(TZ_LOCAL)", f"UTC {raw_first} 转 Berlin", str(converted_first)),
    (6, "return frame", "返回转换后的 DataFrame，第一行本地 timestamp 如右", str(actual_frame.loc[0, "timestamp"])),
]
show_steps("_read_csv 的逐步演算", 'name="household.csv"', rows, actual_frame.head(3))


_read_csv 的逐步演算
输入：name="household.csv"


,步骤,对应源码/表达式,怎么算,结果
0,1,path = self.data_dir / name,tmp\test_prosumer_demo\processed\prosumer / household.csv,tmp\test_prosumer_demo\processed\prosumer\household.csv
1,2,if not path.exists(),文件存在，所以不报 FileNotFoundError,True
2,3,frame = pd.read_csv(path),读入前 1 行 timestamp,2019-12-31 21:00:00+00:00
3,4,if 'timestamp' not in frame.columns,列名包含 timestamp,True
4,5,"pd.to_datetime(..., utc=True).dt.tz_convert(TZ_LOCAL)",UTC 2019-12-31 21:00:00+00:00 转 Berlin,2019-12-31 22:00:00+01:00
5,6,return frame,返回转换后的 DataFrame，第一行本地 timestamp 如右,2019-12-31 22:00:00+01:00


最终输出：


,timestamp,profile_a,profile_b
0,2019-12-31 22:00:00+01:00,1.00,1.500
1,2019-12-31 22:15:00+01:00,1.01,1.515
2,2019-12-31 22:30:00+01:00,1.02,1.530


## 6. `ProsumerDataset._build_split_masks(timestamps)`

作用：判断哪些行是当前 split 的主体数据 `active`，哪些行只是为了补历史而允许读取 `accessible`。

这里用 3 个具体日期演算：`2020-01-01`、`2020-01-02`、`2020-01-03`。配置要求 active 只取 `2020-01-02`，但 `history_warmup_steps=1`，所以同年数据都 accessible。

In [7]:
mask_probe = object.__new__(ProsumerDataset)
mask_probe.year = 2020
mask_probe.start_date = date(2020, 1, 2)
mask_probe.end_date = date(2020, 1, 2)
mask_probe.history_warmup_steps = 1
mask_timestamps = pd.Series(pd.to_datetime(["2020-01-01 12:00", "2020-01-02 12:00", "2020-01-03 12:00"]).tz_localize(TZ_LOCAL))

local_dates = mask_timestamps.dt.date
year_mask = (mask_timestamps.dt.year == mask_probe.year).to_numpy(dtype=bool)
active_mask = year_mask.copy()
active_after_start = active_mask & (local_dates >= mask_probe.start_date).to_numpy(dtype=bool)
active_after_end = active_after_start & (local_dates <= mask_probe.end_date).to_numpy(dtype=bool)
accessible_mask = year_mask if mask_probe.history_warmup_steps > 0 else active_after_end.copy()
actual_active, actual_accessible = mask_probe._build_split_masks(mask_timestamps)

rows = [
    (1, "local_dates = timestamps.dt.date", "从 3 个时间戳取自然日", [str(d) for d in local_dates]),
    (2, "year_mask = timestamps.dt.year == self.year", "三行都是 2020 年", year_mask.tolist()),
    (3, "active_mask = year_mask.copy()", "active 先等于全年 mask", active_mask.tolist()),
    (4, "active_mask &= local_dates >= start_date", "start_date=1/2，1/1 被过滤", active_after_start.tolist()),
    (5, "active_mask &= local_dates <= end_date", "end_date=1/2，1/3 被过滤", active_after_end.tolist()),
    (6, "accessible = year_mask if history_warmup_steps > 0 else active", "history=1，所以同年三天都可读来补历史", accessible_mask.tolist()),
    (7, "return active_mask, accessible_mask", "函数真实输出", {"active": actual_active.tolist(), "accessible": actual_accessible.tolist()}),
]
show_steps("_build_split_masks 的逐步演算", "dates=[1/1,1/2,1/3], start=end=1/2, history=1", rows)


_build_split_masks 的逐步演算
输入：dates=[1/1,1/2,1/3], start=end=1/2, history=1


,步骤,对应源码/表达式,怎么算,结果
0,1,local_dates = timestamps.dt.date,从 3 个时间戳取自然日,"[2020-01-01, 2020-01-02, 2020-01-03]"
1,2,year_mask = timestamps.dt.year == self.year,三行都是 2020 年,"[True, True, True]"
2,3,active_mask = year_mask.copy(),active 先等于全年 mask,"[True, True, True]"
3,4,active_mask &= local_dates >= start_date,start_date=1/2，1/1 被过滤,"[False, True, True]"
4,5,active_mask &= local_dates <= end_date,end_date=1/2，1/3 被过滤,"[False, True, False]"
5,6,accessible = year_mask if history_warmup_steps > 0 else active,history=1，所以同年三天都可读来补历史,"[True, True, True]"
6,7,"return active_mask, accessible_mask",函数真实输出,"{'active': [False, True, False], 'accessible': [True, True, True]}"


## 7. `ProsumerDataset._load_components()`

作用：读取多个负荷分量，检查时间轴一致，然后逐点相加并乘 `load_scale`。

这里看 `2020-01-01 00:00` 这一行的 profile_a/profile_b 怎么算。

In [8]:
load_frame, component_frames, active_on_accessible, source_positions = dataset._load_components()
base_ts = pd.Timestamp("2020-01-01 00:00", tz=TZ_LOCAL)
row_idx = int(load_frame.index[load_frame["timestamp"] == base_ts][0])

h_a = float(component_frames["household"].loc[row_idx, "profile_a"])
hp_a = float(component_frames["heatpump"].loc[row_idx, "profile_a"])
scale_a = float(dataset.load_scale[0])
manual_a = (h_a + hp_a) * scale_a
actual_a = float(load_frame.loc[row_idx, "profile_a"])

h_b = float(component_frames["household"].loc[row_idx, "profile_b"])
hp_b = float(component_frames["heatpump"].loc[row_idx, "profile_b"])
scale_b = float(dataset.load_scale[1])
manual_b = (h_b + hp_b) * scale_b
actual_b = float(load_frame.loc[row_idx, "profile_b"])

rows = [
    (1, "raw_frame = self._read_csv('household.csv')", "读 household，00:00 的 profile_a/profile_b", [h_a, h_b]),
    (2, "raw_frame = self._read_csv('heatpump.csv')", "读 heatpump，00:00 的 profile_a/profile_b", [hp_a, hp_b]),
    (3, "active_mask, accessible_mask = _build_split_masks(...)", "当前 dataset history=2，所以 2020 同年数据可 accessible", {"active_count": int(active_on_accessible.sum()), "accessible_count": len(active_on_accessible)}),
    (4, "frame = raw_frame.loc[accessible_mask, ['timestamp', *agent_profiles]]", "只保留 timestamp、profile_a、profile_b", list(component_frames["household"].columns)),
    (5, "base_timestamp.equals(frame['timestamp'])", "household 和 heatpump 时间轴一致，才能逐点相加", True),
    (6, "total_load = household + heatpump", f"profile_a: {h_a:.3f}+{hp_a:.3f}", round(h_a + hp_a, 6)),
    (7, "total_load * load_scale", f"profile_a: ({h_a:.3f}+{hp_a:.3f})*{scale_a}", round(manual_a, 6)),
    (8, "total_load * load_scale", f"profile_b: ({h_b:.3f}+{hp_b:.3f})*{scale_b}", round(manual_b, 6)),
    (9, "return merged", "函数真实输出 load_frame 在该行的两个值", [round(actual_a, 6), round(actual_b, 6)]),
]
show_steps("_load_components 的逐步演算", "timestamp=2020-01-01 00:00，load_scale=[1.0,10.0]", rows, load_frame.loc[[row_idx]])


_load_components 的逐步演算
输入：timestamp=2020-01-01 00:00，load_scale=[1.0,10.0]


,步骤,对应源码/表达式,怎么算,结果
0,1,raw_frame = self._read_csv('household.csv'),读 household，00:00 的 profile_a/profile_b,"[1.08, 1.62]"
1,2,raw_frame = self._read_csv('heatpump.csv'),读 heatpump，00:00 的 profile_a/profile_b,"[0.216, 0.324]"
2,3,"active_mask, accessible_mask = _build_split_masks(...)",当前 dataset history=2，所以 2020 同年数据可 accessible,"{'active_count': 96, 'accessible_count': 104}"
3,4,"frame = raw_frame.loc[accessible_mask, ['timestamp', *agent_profiles]]",只保留 timestamp、profile_a、profile_b,"[timestamp, profile_a, profile_b]"
4,5,base_timestamp.equals(frame['timestamp']),household 和 heatpump 时间轴一致，才能逐点相加,True
5,6,total_load = household + heatpump,profile_a: 1.080+0.216,1.296
6,7,total_load * load_scale,profile_a: (1.080+0.216)*1.0,1.296
7,8,total_load * load_scale,profile_b: (1.620+0.324)*10.0,19.44
8,9,return merged,函数真实输出 load_frame 在该行的两个值,"[1.296, 19.440001]"


最终输出：


,timestamp,profile_a,profile_b
0,2020-01-01 00:00:00+01:00,1.296,19.440001


## 8. `_aligned_numeric_column(name, column, base_timestamps, label=...)`

作用：把 price/PV 这种单列数值信号，对齐到负荷的时间轴上。

这里用价格列演算：取 `base_timestamps` 的前 3 个时间点，看看 price 怎么被重排和转成数组。

In [9]:
base_timestamps = load_frame["timestamp"].reset_index(drop=True)
name = "price.csv"
column = "price"
raw_price = dataset._read_csv(name)
_, accessible_mask_price = dataset._build_split_masks(raw_price["timestamp"])
filtered_price = raw_price.loc[accessible_mask_price]
indexed_price = filtered_price.set_index("timestamp").sort_index()
reindexed_price = indexed_price.reindex(base_timestamps)
values = pd.to_numeric(reindexed_price[column], errors="coerce").to_numpy(dtype=np.float32)
has_nan = bool(np.isnan(values).any())
actual_price = dataset._aligned_numeric_column(name, column, base_timestamps, label="price demo")

preview = pd.DataFrame(
    {
        "base_timestamp": base_timestamps.astype(str).head(3),
        "price_after_reindex": values[:3],
        "actual_return": actual_price[:3],
    }
)
rows = [
    (1, "frame = self._read_csv('price.csv')", "读入 price.csv，第一行 price", round(float(raw_price.loc[0, "price"]), 6)),
    (2, "_, accessible_mask = _build_split_masks(frame['timestamp'])", "当前同年 accessible 行数", int(accessible_mask_price.sum())),
    (3, "frame.loc[accessible_mask]", "只留下 accessible 行", len(filtered_price)),
    (4, "set_index('timestamp').sort_index().reindex(base_timestamps)", "按负荷时间轴重排，前 3 个价格见输出表", values[:3].round(6).tolist()),
    (5, "pd.to_numeric(...).to_numpy(float32)", "转成 float32 数组", str(values.dtype)),
    (6, "if np.isnan(values).any()", "没有缺失对齐点，所以 False", has_nan),
    (7, "return values", "真实返回数组前 3 个值", actual_price[:3].round(6).tolist()),
]
show_steps("_aligned_numeric_column 的逐步演算", 'name="price.csv", column="price", base_timestamps=load_frame 时间轴', rows, preview)


_aligned_numeric_column 的逐步演算
输入：name="price.csv", column="price", base_timestamps=load_frame 时间轴


,步骤,对应源码/表达式,怎么算,结果
0,1,frame = self._read_csv('price.csv'),读入 price.csv，第一行 price,0.2
1,2,"_, accessible_mask = _build_split_masks(frame['timestamp'])",当前同年 accessible 行数,104
2,3,frame.loc[accessible_mask],只留下 accessible 行,104
3,4,set_index('timestamp').sort_index().reindex(base_timestamps),按负荷时间轴重排，前 3 个价格见输出表,"[0.2199999988079071, 0.22261600196361542, 0.22522099316120148]"
4,5,pd.to_numeric(...).to_numpy(float32),转成 float32 数组,float32
5,6,if np.isnan(values).any(),没有缺失对齐点，所以 False,False
6,7,return values,真实返回数组前 3 个值,"[0.2199999988079071, 0.22261600196361542, 0.22522099316120148]"


最终输出：


,base_timestamp,price_after_reindex,actual_return
0,2020-01-01 00:00:00+01:00,0.220000,0.220000
1,2020-01-01 00:15:00+01:00,0.222616,0.222616
2,2020-01-01 00:30:00+01:00,0.225221,0.225221


## 9. `ProsumerDataset._load_pv(base_timestamps)`

作用：把一条参考 PV 曲线换算成每个智能体自己的 PV 序列。

这里用峰值那一行来手算。`ref_south` 的峰值约是 3.0，`pv_capacity_kw=[5.0,3.0]`，所以：

- profile_a 缩放比例是 `5.0 / 3.0`。
- profile_b 缩放比例是 `3.0 / 3.0`。

In [10]:
reference = dataset._aligned_numeric_column("pv_reference.csv", f"ref_{dataset.pv_reference}", base_timestamps, label="PV reference")
ref_peak_kw = float(np.max(reference))
peak_idx = int(np.argmax(reference))
capacity = np.asarray(dataset.pv_capacity_kw, dtype=np.float32)
capacity_scale = capacity / np.float32(ref_peak_kw)
manual_pv_peak = reference[peak_idx] * capacity_scale
manual_pv_after_scale = manual_pv_peak * dataset.pv_scale
manual_peak_kw = capacity * dataset.pv_scale
actual_pv, actual_pv_peak_kw = dataset._load_pv(base_timestamps)

rows = [
    (1, "reference = _aligned_numeric_column('pv_reference.csv', 'ref_south', ...)", "参考曲线峰值位置的 ref_south", round(float(reference[peak_idx]), 6)),
    (2, "ref_peak_kw = np.max(reference)", "reference 最大值", round(ref_peak_kw, 6)),
    (3, "capacity = np.asarray(pv_capacity_kw)", "配置里两个智能体容量", capacity.tolist()),
    (4, "scales = capacity / ref_peak_kw", f"[5,3] / {ref_peak_kw:.6f}", capacity_scale.round(6).tolist()),
    (5, "pv = reference[:,None] * scales[None,:]", f"峰值行：{ref_peak_kw:.6f} * scales", manual_pv_peak.round(6).tolist()),
    (6, "pv = pv * pv_scale", "pv_scale=[1,1]，数值不变", manual_pv_after_scale.round(6).tolist()),
    (7, "pv_peak_kw = capacity * pv_scale", "峰值容量也乘 pv_scale", manual_peak_kw.round(6).tolist()),
    (8, "return pv, pv_peak_kw", "真实输出峰值行和峰值容量", {"pv_peak_row": actual_pv[peak_idx].round(6).tolist(), "pv_peak_kw": actual_pv_peak_kw.round(6).tolist()}),
]
show_steps("_load_pv 的逐步演算", "ref_south 峰值行；pv_capacity_kw=[5.0,3.0], pv_scale=1.0", rows)


_load_pv 的逐步演算
输入：ref_south 峰值行；pv_capacity_kw=[5.0,3.0], pv_scale=1.0


,步骤,对应源码/表达式,怎么算,结果
0,1,"reference = _aligned_numeric_column('pv_reference.csv', 'ref_south', ...)",参考曲线峰值位置的 ref_south,3.0
1,2,ref_peak_kw = np.max(reference),reference 最大值,3.0
2,3,capacity = np.asarray(pv_capacity_kw),配置里两个智能体容量,"[5.0, 3.0]"
3,4,scales = capacity / ref_peak_kw,"[5,3] / 3.000000","[1.6666669845581055, 1.0]"
4,5,"pv = reference[:,None] * scales[None,:]",峰值行：3.000000 * scales,"[5.0, 3.0]"
5,6,pv = pv * pv_scale,"pv_scale=[1,1]，数值不变","[5.0, 3.0]"
6,7,pv_peak_kw = capacity * pv_scale,峰值容量也乘 pv_scale,"[5.0, 3.0]"
7,8,"return pv, pv_peak_kw",真实输出峰值行和峰值容量,"{'pv_peak_row': [5.0, 3.0], 'pv_peak_kw': [5.0, 3.0]}"


## 10. `ProsumerDataset._append_episode_windows(...)`

作用：把连续 active 索引切成固定长度 episode，并记录 history 和 bootstrap 的边界。

这里不用真实 CSV，直接用最小数学例子：连续 active 区间是 `[0,8)`，episode 长度 3，步长 2，历史长度 1。

In [11]:
scratch = object.__new__(ProsumerDataset)
scratch.episode_length = 3
scratch.window_stride_steps = 2
scratch.history_warmup_steps = 1
scratch._episode_slices = []
scratch.dropped_tail_steps = 0

run_start_idx = 0
run_end_idx = 8
segment_start_idx = 0
first_usable = max(run_start_idx, segment_start_idx + scratch.history_warmup_steps)
usable_steps = run_end_idx - first_usable
starts = list(range(first_usable, run_end_idx - scratch.episode_length + 1, scratch.window_stride_steps))
manual_slices = []
for start in starts:
    end = start + scratch.episode_length
    next_active = end if end < run_end_idx else None
    manual_slices.append((start - scratch.history_warmup_steps, start, end, next_active))
manual_dropped_tail = max(run_end_idx - (starts[-1] + scratch.episode_length), 0)

scratch._append_episode_windows(run_start_idx=run_start_idx, run_end_idx=run_end_idx, segment_start_idx=segment_start_idx)

rows = [
    (1, "first_usable = max(run_start_idx, segment_start_idx + history)", "max(0,0+1)", first_usable),
    (2, "usable_steps = run_end_idx - first_usable", "8 - 1", usable_steps),
    (3, "if usable_steps < episode_length", "7 < 3 为 False，继续切 episode", False),
    (4, "starts = range(first_usable, run_end-episode+1, stride)", "range(1, 8-3+1, 2)", starts),
    (5, "第 1 个 start=1", "end=1+3=4, next_active=4, history_start=1-1=0", manual_slices[0]),
    (6, "第 2 个 start=3", "end=3+3=6, next_active=6, history_start=3-1=2", manual_slices[1]),
    (7, "第 3 个 start=5", "end=5+3=8, end 不小于 run_end，所以 next_active=None", manual_slices[2]),
    (8, "dropped_tail_steps += max(run_end - (last_start + episode), 0)", "max(8-(5+3),0)", manual_dropped_tail),
    (9, "真实 _episode_slices", "调用函数后的内部切片", scratch._episode_slices),
]
show_steps("_append_episode_windows 的逐步演算", "run=[0,8), episode=3, stride=2, history=1", rows)


_append_episode_windows 的逐步演算
输入：run=[0,8), episode=3, stride=2, history=1


,步骤,对应源码/表达式,怎么算,结果
0,1,"first_usable = max(run_start_idx, segment_start_idx + history)","max(0,0+1)",1
1,2,usable_steps = run_end_idx - first_usable,8 - 1,7
2,3,if usable_steps < episode_length,7 < 3 为 False，继续切 episode,False
3,4,"starts = range(first_usable, run_end-episode+1, stride)","range(1, 8-3+1, 2)","[1, 3, 5]"
4,5,第 1 个 start=1,"end=1+3=4, next_active=4, history_start=1-1=0","(0, 1, 4, 4)"
5,6,第 2 个 start=3,"end=3+3=6, next_active=6, history_start=3-1=2","(2, 3, 6, 6)"
6,7,第 3 个 start=5,"end=5+3=8, end 不小于 run_end，所以 next_active=None","(4, 5, 8, None)"
7,8,"dropped_tail_steps += max(run_end - (last_start + episode), 0)","max(8-(5+3),0)",0
8,9,真实 _episode_slices,调用函数后的内部切片,"[(0, 1, 4, 4), (2, 3, 6, 6), (4, 5, 8, None)]"


## 11. `ProsumerDataset._load()`

作用：完整加载流程：先建立总负荷时间轴，再对齐 price/PV，最后生成 meta、normalization 位置和 episode 切片。

这个方法在 `__init__` 里自动执行。下面用已经创建好的 `dataset` 复盘关键计算。

In [12]:
# 重新计算 _load 内部关键中间值，和 dataset 已保存结果对比。
load_frame_again, component_frames_again, active_mask_again, source_positions_again = dataset._load_components()
base_timestamps_again = load_frame_again["timestamp"].reset_index(drop=True)
wholesale_price_again = dataset._aligned_numeric_column("price.csv", "price", base_timestamps_again, label="price")
pv_again, pv_peak_kw_again = dataset._load_pv(base_timestamps_again)
load_again = load_frame_again.loc[:, dataset.agent_profiles].to_numpy(dtype=np.float32)
shape_matches = load_again.shape == pv_again.shape
active_positions = np.flatnonzero(active_mask_again)
split_points = np.flatnonzero(np.diff(source_positions_again) != 1) + 1
bounds = np.concatenate([[0], split_points, [len(source_positions_again)]])

sample_i = int(active_positions[0])
rows = [
    (1, "load_frame, component_frames, active_mask, source_positions = _load_components()", "得到总负荷和 active/source 信息", {"load_rows": len(load_frame_again), "active_rows": int(active_mask_again.sum())}),
    (2, "base_timestamps = load_frame['timestamp']", "负荷时间轴第一个 active 索引", str(base_timestamps_again.iloc[sample_i])),
    (3, "wholesale_price = _aligned_numeric_column('price.csv', 'price', base_timestamps)", f"同一行价格", round(float(wholesale_price_again[sample_i]), 6)),
    (4, "pv, pv_peak_kw = _load_pv(base_timestamps)", f"同一行 PV 两个智能体", pv_again[sample_i].round(6).tolist()),
    (5, "load = load_frame[agent_profiles].to_numpy(float32)", f"同一行负荷两个智能体", load_again[sample_i].round(6).tolist()),
    (6, "if load.shape != pv.shape", f"load.shape={load_again.shape}, pv.shape={pv_again.shape}", shape_matches),
    (7, "self._signals = {'price','load','pv'} + load components", "保存的信号 key", list(dataset._signals.keys())),
    (8, "self._normalization_positions = active_positions", "前 5 个 active 位置", active_positions[:5].tolist()),
    (9, "split_points = np.flatnonzero(np.diff(source_positions) != 1)+1", "source 连续，所以没有断点", split_points.tolist()),
    (10, "bounds = [0] + split_points + [len(source_positions)]", "连续段边界", bounds.tolist()),
    (11, "_append_episode_windows(...)", "最终 episode 数量", dataset.num_episodes()),
]
show_steps("_load 的逐步演算", "使用完整示例 CSV；复盘 dataset 初始化时的加载过程", rows)


_load 的逐步演算
输入：使用完整示例 CSV；复盘 dataset 初始化时的加载过程


,步骤,对应源码/表达式,怎么算,结果
0,1,"load_frame, component_frames, active_mask, source_positions = _load_components()",得到总负荷和 active/source 信息,"{'load_rows': 104, 'active_rows': 96}"
1,2,base_timestamps = load_frame['timestamp'],负荷时间轴第一个 active 索引,2020-01-01 00:00:00+01:00
2,3,"wholesale_price = _aligned_numeric_column('price.csv', 'price', base_timestamps)",同一行价格,0.22
3,4,"pv, pv_peak_kw = _load_pv(base_timestamps)",同一行 PV 两个智能体,"[0.0, 0.0]"
4,5,load = load_frame[agent_profiles].to_numpy(float32),同一行负荷两个智能体,"[1.2960000038146973, 19.440000534057617]"
5,6,if load.shape != pv.shape,"load.shape=(104, 2), pv.shape=(104, 2)",True
6,7,"self._signals = {'price','load','pv'} + load components",保存的信号 key,"[wholesale_price, load, pv, load_household, load_heatpump]"
7,8,self._normalization_positions = active_positions,前 5 个 active 位置,"[0, 1, 2, 3, 4]"
8,9,split_points = np.flatnonzero(np.diff(source_positions) != 1)+1,source 连续，所以没有断点,[]
9,10,bounds = [0] + split_points + [len(source_positions)],连续段边界,"[0, 104]"


## 12. `ProsumerDataset.num_episodes()`

作用：返回当前数据集里已经切好的 episode 数量。

In [13]:
manual_len = len(dataset._episode_slices)
actual_num = dataset.num_episodes()
rows = [
    (1, "self._episode_slices", "内部已经保存的 episode 切片，前 3 个如下", dataset._episode_slices[:3]),
    (2, "self._num_episodes = len(self._episode_slices)", f"len(_episode_slices)={manual_len}", manual_len),
    (3, "return self._num_episodes", "真实返回值", actual_num),
]
show_steps("num_episodes 的逐步演算", "当前 dataset 已加载完成", rows, actual_num)


num_episodes 的逐步演算
输入：当前 dataset 已加载完成


,步骤,对应源码/表达式,怎么算,结果
0,1,self._episode_slices,内部已经保存的 episode 切片，前 3 个如下,"[(0, 2, 10, 10), (8, 10, 18, 18), (16, 18, 26, 26)]"
1,2,self._num_episodes = len(self._episode_slices),len(_episode_slices)=11,11
2,3,return self._num_episodes,真实返回值,11


最终输出：


11

## 13. `ProsumerDataset.get_normalization_signal_values(signal_name)`

作用：只取 active 行的信号样本，给归一化器拟合统计量。

下面用 `load` 演算：先找 active 索引，再取 `_signals['load']` 中这些位置。

In [14]:
signal_name = "load"
signal_exists = signal_name in dataset._signals
positions = dataset._normalization_positions
first_positions = positions[:3]
manual_values = np.asarray(dataset._signals[signal_name][positions], dtype=np.float32)
actual_values = dataset.get_normalization_signal_values(signal_name)

rows = [
    (1, "if signal_name not in self._signals", 'signal_name="load" 存在', not signal_exists),
    (2, "self._normalization_positions", "active 区间的位置，前 3 个", first_positions.tolist()),
    (3, "self._signals['load'][positions]", "按 active 位置取负荷，前 3 行", manual_values[:3].round(6).tolist()),
    (4, "np.asarray(..., dtype=float32)", "转成 float32 返回，前 3 行", actual_values[:3].round(6).tolist()),
]
show_steps("get_normalization_signal_values 的逐步演算", 'signal_name="load"', rows)


get_normalization_signal_values 的逐步演算
输入：signal_name="load"


,步骤,对应源码/表达式,怎么算,结果
0,1,if signal_name not in self._signals,"signal_name=""load"" 存在",False
1,2,self._normalization_positions,active 区间的位置，前 3 个,"[0, 1, 2]"
2,3,self._signals['load'][positions],按 active 位置取负荷，前 3 行,"[[1.2960000038146973, 19.440000534057617], [1.3079999685287476, 19.6200008392334], [1.3200000524520874, 19.799999237..."
3,4,"np.asarray(..., dtype=float32)",转成 float32 返回，前 3 行,"[[1.2960000038146973, 19.440000534057617], [1.3079999685287476, 19.6200008392334], [1.3200000524520874, 19.799999237..."


## 14. `ProsumerDataset._signal_slice(start, end)`

作用：从所有信号里截取同一个 `[start, end)` 时间窗口，并返回副本。

这里取 `start=2, end=5`，所以应该拿索引 2、3、4。

In [15]:
start = 2
end = 5
manual_load = dataset._signals["load"][start:end].copy()
manual_pv = dataset._signals["pv"][start:end].copy()
actual_slice = dataset._signal_slice(start, end)

rows = [
    (1, "for name, values in self._signals.items()", "遍历所有信号 key", list(dataset._signals.keys())),
    (2, "values[start:end].copy()", "load 取索引 2,3,4 的 profile_a/profile_b", manual_load.round(6).tolist()),
    (3, "values[start:end].copy()", "pv 取索引 2,3,4 的 profile_a/profile_b", manual_pv.round(6).tolist()),
    (4, "return {name: ...}", "真实返回中 load/pv 两项", {"load": actual_slice["load"].round(6).tolist(), "pv": actual_slice["pv"].round(6).tolist()}),
]
show_steps("_signal_slice 的逐步演算", "start=2, end=5，也就是取 [2,3,4]", rows)


_signal_slice 的逐步演算
输入：start=2, end=5，也就是取 [2,3,4]


,步骤,对应源码/表达式,怎么算,结果
0,1,"for name, values in self._signals.items()",遍历所有信号 key,"[wholesale_price, load, pv, load_household, load_heatpump]"
1,2,values[start:end].copy(),"load 取索引 2,3,4 的 profile_a/profile_b","[[1.3200000524520874, 19.799999237060547], [1.3320000171661377, 19.979999542236328], [1.343999981880188, 20.15999984..."
2,3,values[start:end].copy(),"pv 取索引 2,3,4 的 profile_a/profile_b","[[0.0, 0.0], [0.0, 0.0], [0.0, 0.0]]"
3,4,return {name: ...},真实返回中 load/pv 两项,"{'load': [[1.3200000524520874, 19.799999237060547], [1.3320000171661377, 19.979999542236328], [1.343999981880188, 20..."


## 15. `ProsumerDataset.get_episode(episode_idx)`

作用：导出一个完整 episode 数据包，包括：

- `history_signals`：active 前面的历史预热。
- `signals`：真正给环境 step 的 active 区间。
- `bootstrap_signals`：下一步目标/引导值，如果存在。
- `meta`：索引、时间戳、单位和配置。

下面用第 0 个 episode 演算。

In [16]:
episode_idx = 0
in_range = 0 <= episode_idx < dataset._num_episodes
history_start, active_start, active_end, next_active = dataset._episode_slices[episode_idx]
timestamps = dataset._timestamps.iloc[active_start:active_end].reset_index(drop=True)
history_timestamps = dataset._timestamps.iloc[history_start:active_start].reset_index(drop=True)
bootstrap_timestamps = [] if next_active is None else dataset._timestamps.iloc[next_active:next_active+1].astype(str).tolist()
manual_signals = dataset._signal_slice(active_start, active_end)
manual_history = dataset._signal_slice(history_start, active_start)
manual_bootstrap = {} if next_active is None else dataset._signal_slice(next_active, next_active + 1)
episode = dataset.get_episode(episode_idx)

rows = [
    (1, "if episode_idx < 0 or episode_idx >= self._num_episodes", f"0 <= {episode_idx} < {dataset._num_episodes}", in_range),
    (2, "history_start, start, end, next_active = self._episode_slices[episode_idx]", "第 0 个切片", (history_start, active_start, active_end, next_active)),
    (3, "timestamps = self._timestamps.iloc[start:end]", f"active 索引 [{active_start},{active_end})", timestamps.astype(str).tolist()),
    (4, "history_timestamps = self._timestamps.iloc[history_start:start]", f"history 索引 [{history_start},{active_start})", history_timestamps.astype(str).tolist()),
    (5, "bootstrap_timestamps", f"next_active={next_active}，取 1 行 bootstrap", bootstrap_timestamps),
    (6, "signals = _signal_slice(start,end)", "active load 第一行", manual_signals["load"][0].round(6).tolist()),
    (7, "history_signals = _signal_slice(history_start,start)", "history load 所有行", manual_history["load"].round(6).tolist()),
    (8, "bootstrap_signals = _signal_slice(next_active,next_active+1)", "bootstrap load", manual_bootstrap["load"].round(6).tolist() if manual_bootstrap else {}),
    (9, "history_length = int(start - history_start)", f"{active_start} - {history_start}", episode["history_length"]),
    (10, "meta = {...}", "meta 中关键索引", {k: episode["meta"][k] for k in ["episode_idx", "history_start_idx", "active_start_idx", "active_end_idx", "next_active_idx"]}),
]
show_steps("get_episode 的逐步演算", "episode_idx=0", rows)

print("最终 episode 顶层 key：", list(episode.keys()))
print("最终 signals key：", list(episode["signals"].keys()))


get_episode 的逐步演算
输入：episode_idx=0


,步骤,对应源码/表达式,怎么算,结果
0,1,if episode_idx < 0 or episode_idx >= self._num_episodes,0 <= 0 < 11,True
1,2,"history_start, start, end, next_active = self._episode_slices[episode_idx]",第 0 个切片,"(0, 2, 10, 10)"
2,3,timestamps = self._timestamps.iloc[start:end],"active 索引 [2,10)","[2020-01-01 00:30:00+01:00, 2020-01-01 00:45:00+01:00, 2020-01-01 01:00:00+01:00, 2020-01-01 01:15:00+01:00, 2020-01..."
3,4,history_timestamps = self._timestamps.iloc[history_start:start],"history 索引 [0,2)","[2020-01-01 00:00:00+01:00, 2020-01-01 00:15:00+01:00]"
4,5,bootstrap_timestamps,next_active=10，取 1 行 bootstrap,[2020-01-01 02:30:00+01:00]
5,6,"signals = _signal_slice(start,end)",active load 第一行,"[1.3200000524520874, 19.799999237060547]"
6,7,"history_signals = _signal_slice(history_start,start)",history load 所有行,"[[1.2960000038146973, 19.440000534057617], [1.3079999685287476, 19.6200008392334]]"
7,8,"bootstrap_signals = _signal_slice(next_active,next_active+1)",bootstrap load,"[[1.4160000085830688, 21.240001678466797]]"
8,9,history_length = int(start - history_start),2 - 0,2
9,10,meta = {...},meta 中关键索引,"{'episode_idx': 0, 'history_start_idx': 0, 'active_start_idx': 2, 'active_end_idx': 10, 'next_active_idx': 10}"


最终 episode 顶层 key： ['signals', 'history_signals', 'bootstrap_signals', 'history_timestamps', 'history_length', 'meta']
最终 signals key： ['wholesale_price', 'load', 'pv', 'load_household', 'load_heatpump']


## 16. 总结：调用链怎么串起来

这张表把上面所有函数放回主流程里。你可以从上到下看：先把配置标准化，再读 CSV，再分 active/accessible，再对齐信号，最后切 episode 并导出。

In [17]:
flow = pd.DataFrame(
    [
        ("_parse_optional_local_date", "配置日期", '"2020-01-01 15:30" -> date(2020,1,1)', "把日期输入统一成自然日"),
        ("_coerce_scale_vector", "缩放配置", "scale=2,n_agents=3 -> [2,2,2]", "让倍率能和智能体列相乘"),
        ("__init__", "入口", "household=1.0, heatpump=0.2, scale=10 -> load=12.0", "创建可用数据集对象"),
        ("_validate_init_args", "参数校验", "episode_length=0 -> ValueError", "提前拦住非法配置"),
        ("_read_csv", "读取 CSV", "UTC 23:00 -> Berlin 00:00", "统一时间戳口径"),
        ("_build_split_masks", "筛行", "[1/1,1/2,1/3], start=end=1/2 -> active=[F,T,F]", "区分主体数据和历史上下文"),
        ("_load_components", "负荷合成", "(1.0+0.2)*10=12.0", "合成总负荷"),
        ("_aligned_numeric_column", "数值对齐", "base=[t0,t1], price={t0:0.2,t1:0.3} -> [0.2,0.3]", "避免 price/PV 和 load 错位"),
        ("_load_pv", "PV 缩放", "ref_peak=3, capacity=6 -> 峰值 PV=6", "把参考曲线转成智能体 PV"),
        ("_append_episode_windows", "切 episode", "run=[0,8), episode=3,stride=2,history=1 -> 3 个切片", "把连续时间序列切成训练单元"),
        ("_load", "总控加载", "load/price/PV 在同一 t0 保存", "构建内部缓存"),
        ("num_episodes", "计数", "3 个切片 -> 3", "告诉外部有多少 episode"),
        ("get_normalization_signal_values", "归一化样本", "load=[10,20,30], pos=[0,2] -> [10,30]", "只用 active 样本拟合统计量"),
        ("_signal_slice", "切信号", "load=[10,20,30], start=1,end=3 -> [20,30]", "给 episode 导出同一时间窗"),
        ("get_episode", "导出 episode", "切片 (0,2,5,5) -> history=[0,1], signals=[2,3,4], bootstrap=[5]", "外部真正消费的数据包"),
    ],
    columns=["函数", "位置", "数学例子", "意义"],
)
display(flow)

,函数,位置,数学例子,意义
0,_parse_optional_local_date,配置日期,"""2020-01-01 15:30"" -> date(2020,1,1)",把日期输入统一成自然日
1,_coerce_scale_vector,缩放配置,"scale=2,n_agents=3 -> [2,2,2]",让倍率能和智能体列相乘
2,__init__,入口,"household=1.0, heatpump=0.2, scale=10 -> load=12.0",创建可用数据集对象
3,_validate_init_args,参数校验,episode_length=0 -> ValueError,提前拦住非法配置
4,_read_csv,读取 CSV,UTC 23:00 -> Berlin 00:00,统一时间戳口径
5,_build_split_masks,筛行,"[1/1,1/2,1/3], start=end=1/2 -> active=[F,T,F]",区分主体数据和历史上下文
6,_load_components,负荷合成,(1.0+0.2)*10=12.0,合成总负荷
7,_aligned_numeric_column,数值对齐,"base=[t0,t1], price={t0:0.2,t1:0.3} -> [0.2,0.3]",避免 price/PV 和 load 错位
8,_load_pv,PV 缩放,"ref_peak=3, capacity=6 -> 峰值 PV=6",把参考曲线转成智能体 PV
9,_append_episode_windows,切 episode,"run=[0,8), episode=3,stride=2,history=1 -> 3 个切片",把连续时间序列切成训练单元
